# 02 · Quantization: from 32-bit floats to 8-bit integers

**Goals of this notebook**

1. Understand how computers store numbers, and why 8-bit integers are attractive on a microcontroller.
2. Learn the quantization formula $r \approx s\,(q - z)$ and compute it by hand.
3. Quantize the weights of the baseline model and measure the error.
4. Quantize the *whole* network, so that it only computes with integers, exactly like the Arduino will.
5. Push the bit width down to 4, 3 and 2 bits, and rescue the accuracy with **quantization-aware training**.
6. Meet the other flavours of quantization that toolkits offer (dynamic range, float16, 16-bit activations, partial quantization, integer inputs and outputs) and learn when to pick which.

In [ ]:
import sys, pathlib, copy
# Only needed so Python can find the course folder `aiedge` (the helpers imported below).
ROOT = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == "notebooks" else pathlib.Path.cwd()
sys.path.insert(0, str(ROOT))

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

# ---------------------------------------------------------------------------
# EXTERNAL (shared with the other notebooks, code shown in notebook 01):
#   load_gesture_data  the synthetic sensor data (see the comment block in notebook 01)
#   MLP, count_params  the small network class and a parameter counter
#   train, accuracy, set_seed   the training loop, test accuracy, repeatable random numbers
# Why external: these are used in every notebook. Notebook 01 explains them line by
# line; repeating them here would only copy code. Everything about quantization,
# which is the topic of THIS notebook, is written out below.
# ---------------------------------------------------------------------------
from aiedge.data import load_gesture_data
from aiedge.models import MLP, count_params
from aiedge.training import train, accuracy, set_seed

First we re-create the data and the baseline from notebook 01. Same seed, so it is the same model; this takes a few seconds.

In [ ]:
data = load_gesture_data()
set_seed(0)
baseline = MLP(hidden=(64, 32))
train(baseline, data, augment=True, verbose=False)
base_acc = accuracy(baseline, data.X_test, data.y_test)
print(f"baseline (float32): test accuracy {base_acc:.1%}, {count_params(baseline):,} parameters "
      f"= {4 * count_params(baseline) / 1024:.1f} KB")

## 1. How computers store numbers

**float32** (a 32-bit floating-point number, `float` in C) can store values like 0.000123 or 45 678.9. It uses 32 bits: 1 sign bit, 8 bits for an exponent (*where is the decimal point?*) and 23 bits for the digits (the *mantissa*). That is very flexible, but it takes 4 bytes, and floating-point arithmetic needs complex hardware.

**int8** (an 8-bit integer, `int8_t` in C) can only store the whole numbers from **-128 to 127**: 256 different values in 1 byte.

Why bother with int8 on a microcontroller?

1. **4x less memory**: 1 byte instead of 4 per weight. This is the big one for us.
2. **Less energy.** Measured on a 45 nm chip (Horowitz, ISSCC 2014):

   | operation | energy |
   |---|---|
   | 8-bit integer add | 0.03 pJ |
   | 32-bit float add | 0.9 pJ |
   | 8-bit integer multiply | 0.2 pJ |
   | 32-bit float multiply | 3.7 pJ |
   | read 32 bits from a small on-chip memory | ~5 pJ |

   Integer maths is ~20x cheaper, and moving fewer bytes around saves energy too.
3. **Speed**: on chips *without* an FPU, float maths is emulated in software and is 10-100x slower. Our Cortex-M4F *has* an FPU, so the gain there comes from special instructions that do two or four 8/16-bit multiplications at once (used by libraries like CMSIS-NN). In notebook 05 you will measure what *our* simple C code achieves.

The question is: **how do we represent numbers like 0.0372 or -1.25 with integers from -128 to 127, without ruining the network?**

## 2. The quantization formula

We pick a **scale** $s$ (a small positive float) and a **zero-point** $z$ (an integer). A real number $r$ is represented by the integer

$$q = \text{clamp}\big(\text{round}(r / s) + z,\ q_{min},\ q_{max}\big)$$

and we can get back (approximately) the real number with

$$r \approx s \cdot (q - z)$$

Think of it as a ruler with 256 marks spaced $s$ apart: every real number is rounded to the nearest mark. The *rounding error* is at most $s/2$, half the distance between marks. The *clamp* cuts off numbers outside the ruler (*clipping error*).

### Symmetric quantization (for weights)
Weights are spread around 0 (both signs). We use $z = 0$ and pick $s$ so that the largest absolute weight lands on 127:

$$s = \frac{\max |w|}{127}$$

Let's do it by hand for four weights:

In [ ]:
w = torch.tensor([-0.62, 0.047, 0.313, 1.27])
s = w.abs().max() / 127                      # scale
q = torch.round(w / s).clamp(-127, 127)      # quantize   -> integers
w_back = s * q                               # dequantize -> approximately w

print(f"scale s = {s:.4f}")
print("original   :", w.numpy())
print("integers q :", q.int().numpy())
print("dequantized:", w_back.numpy().round(4))
print("error      :", (w_back - w).numpy().round(4), f"  (max possible: s/2 = {s / 2:.4f})")

### Asymmetric quantization (for activations)
After a ReLU all values are ≥ 0. Symmetric quantization would waste the whole negative half of the ruler (-128…-1). Instead we map the actual range $[lo, hi]$ onto $[-128, 127]$:

$$s = \frac{hi - lo}{255}, \qquad z = -128 - \text{round}(lo / s)$$

For a ReLU output with range $[0, 6]$: $s = 6/255 \approx 0.0235$ and $z = -128$. So real 0 ↦ -128, real 6 ↦ 127. Note that real **0 is represented exactly** (by $q = z$): this matters, because ReLU produces lots of exact zeros.

In [ ]:
def asymmetric_params(lo, hi, bits=8):
    """Scale and zero-point that map the real range [lo, hi] onto the integers -128...127."""
    lo, hi = min(float(lo), 0.0), max(float(hi), 0.0)   # the range must contain 0 (see above)
    qmin, qmax = -(2 ** (bits - 1)), 2 ** (bits - 1) - 1
    scale = max(hi - lo, 1e-8) / (qmax - qmin)           # real distance between two marks
    zero_point = int(round(qmin - lo / scale))           # the integer that stands for real 0
    return scale, int(np.clip(zero_point, qmin, qmax))

scale, zp = asymmetric_params(0.0, 6.0)
a = torch.tensor([0.0, 0.01, 1.0, 3.3, 6.0])
q = torch.clamp(torch.round(a / scale) + zp, -128, 127)
print(f"scale = {scale:.4f}, zero-point = {zp}")
print("real       :", a.numpy())
print("integers q :", q.int().numpy())
print("dequantized:", (scale * (q - zp)).numpy().round(4))

### The range trade-off
The range must be chosen well:
* **too wide** (e.g. one huge outlier weight) → $s$ is large → all the *normal* values get coarse rounding;
* **too narrow** → large values are clipped.

This is why we often use one scale **per output neuron** ("per-channel") for weights instead of one scale for a whole layer ("per-tensor"): one neuron with large weights then cannot ruin the precision of all the others.

## 3. Quantizing the baseline's weights

Let's look at the weights of the first layer (19 200 numbers):

In [ ]:
W0 = baseline.layers[0].weight.detach()
fig, (a1, a2) = plt.subplots(1, 2, figsize=(11, 3.3))
a1.hist(W0.flatten().numpy(), bins=100, color="C0")
a1.set_xlabel("weight value"); a1.set_ylabel("count"); a1.set_title("Layer 0: all weights")
a2.bar(range(W0.shape[0]), W0.abs().amax(dim=1).numpy(), color="C0")
a2.set_xlabel("neuron (output channel)"); a2.set_ylabel("max |w|"); a2.set_title("Largest weight per neuron")
fig.tight_layout(); plt.show()

Most weights are small, and the largest weight differs quite a lot from neuron to neuron. Let's write a function that quantizes the weights of every layer (per-tensor or per-channel) and puts the *dequantized* values back, so we can measure what the rounding does to the accuracy. This is called **simulated quantization**: still floats, but only values that an integer could represent.

In [ ]:
def quantize_weights(model, bits=8, per_channel=True):
    """Return a copy of `model` whose weights are rounded to `bits`-bit integers (and back)."""
    qmax = 2 ** (bits - 1) - 1                         # 127 for 8 bits, 7 for 4 bits
    m = copy.deepcopy(model)
    with torch.no_grad():
        for layer in m.layers:
            w = layer.weight
            if per_channel:
                max_abs = w.abs().amax(dim=1, keepdim=True)   # one scale per neuron (row)
            else:
                max_abs = w.abs().max()                       # one scale for the layer
            s = max_abs / qmax
            layer.weight.copy_(torch.clamp(torch.round(w / s), -qmax, qmax) * s)
    return m

print(f"{'bits':>5} {'per-tensor':>12} {'per-channel':>12}   (test accuracy, float32 = {base_acc:.1%})")
for bits in [8, 6, 4, 3, 2]:
    acc_t = accuracy(quantize_weights(baseline, bits, per_channel=False), data.X_test, data.y_test)
    acc_c = accuracy(quantize_weights(baseline, bits, per_channel=True), data.X_test, data.y_test)
    print(f"{bits:>5} {acc_t:>12.1%} {acc_c:>12.1%}")

**Observations**
* With **8 bits the accuracy does not change** (differences of a few tenths of a point are noise): the rounding errors are far smaller than what the network cares about. This is why int8 is the industry standard.
* Down to ~4 bits things still work; at 2 bits (only the values -1, 0, +1 times a scale!) the network breaks.
* Down to 4 bits, per-tensor and per-channel perform about the same; at very low bit widths **per-channel is clearly better**.

## 4. Quantizing the whole network: activations too

So far only the *weights* were integers. For the board we want **int8 inference**: the inputs and the activations between the layers must be int8 as well, so that the multiply-accumulates (MACs) can run on integers. For weights we know the range (we can just look at them). Activations depend on the input, so we must **measure** their range by running real data through the network. That is called **calibration**.

The class `QuantMLP` below wraps our model and simulates integer inference:
* input → quantize with (scale, zero-point) of the input;
* each layer → weights quantized per channel, bias quantized to int32;
* after each ReLU → quantize the activation with its own (scale, zero-point).

This approach (train in float, then calibrate and quantize) is called **post-training quantization (PTQ)**.

Before the class we need three tiny helpers that you already know from section 2:

* `quantize`: real number → integer (round, shift by the zero-point, clamp);
* `dequantize`: integer → (approximately) the real number again;
* `fake_quantize`: both in a row. The result is still a float, but it can only be a value an integer could represent. This is how we *simulate* integer maths while staying in PyTorch.

The odd-looking last line of `fake_quantize` (`x_c + (fq - x_c).detach()`) is only needed for training with rounding in section 6. Ignore it for now: it does not change the *value*, only how gradients flow.

In [ ]:
def quantize(x, scale, zero_point, qmin, qmax):
    """real -> integer"""
    return torch.clamp(torch.round(x / scale) + zero_point, qmin, qmax)

def dequantize(q, scale, zero_point):
    """integer -> real (approximately the original number)"""
    return (q - zero_point) * scale

def fake_quantize(x, scale, zero_point, qmin, qmax):
    """Round to the integer grid and back, so the result is a float that only takes 'integer' values."""
    lo, hi = (qmin - zero_point) * scale, (qmax - zero_point) * scale
    x_clipped = torch.maximum(torch.minimum(x, torch.as_tensor(hi)), torch.as_tensor(lo))
    rounded = dequantize(quantize(x_clipped, scale, zero_point, qmin, qmax), scale, zero_point)
    # Same value as `rounded`, but backpropagation treats the rounding as "do nothing"
    # (the straight-through estimator, explained in section 6).
    return x_clipped + (rounded - x_clipped).detach()

Now the wrapper. Read it in three parts:

1. `calibrate` runs the training data through the float network and remembers the smallest and largest value at the input and after every hidden ReLU.
2. `act_params` turns such a range into a (scale, zero-point) pair with `asymmetric_params`.
3. `forward` does what the board will do: quantize the input, then for each layer use quantized weights (one scale per neuron, $z=0$) and a quantized bias, apply ReLU, and quantize the result again.

In [ ]:
class QuantMLP(nn.Module):
    """Wraps a trained MLP and simulates integer inference with fake quantization."""

    def __init__(self, mlp, weight_bits=8, act_bits=8, ema_momentum=0.02):
        super().__init__()
        self.mlp = mlp
        self.weight_bits, self.act_bits = weight_bits, act_bits
        self.ema_momentum = ema_momentum              # only used while training (section 6)
        n_points = len(mlp.layers)                    # the input + every hidden layer output
        self.register_buffer("act_lo", torch.zeros(n_points))   # smallest value seen at each point
        self.register_buffer("act_hi", torch.zeros(n_points))   # largest value seen at each point
        self.calibrated = False

    @torch.no_grad()
    def calibrate(self, X):
        """Measure the range of the input and of every hidden activation on real data."""
        self.mlp.eval()
        x = X
        self.act_lo[0], self.act_hi[0] = x.min(), x.max()
        for i, layer in enumerate(self.mlp.layers[:-1]):
            x = torch.relu(layer(x))
            self.act_lo[i + 1], self.act_hi[i + 1] = 0.0, x.max()   # after ReLU the minimum is 0
        self.calibrated = True

    def act_params(self, k):
        """(scale, zero-point) of measurement point k (0 = input, 1 = after hidden layer 1, ...)."""
        return asymmetric_params(self.act_lo[k].item(), self.act_hi[k].item(), self.act_bits)

    def weight_scale(self, layer):
        """One scale per output neuron: its largest |weight| lands on the largest integer."""
        qmax = 2 ** (self.weight_bits - 1) - 1
        return torch.clamp(layer.weight.abs().amax(dim=1).detach(), min=1e-8) / qmax

    def _act(self, k, x):
        """Fake-quantize an activation. While training, the ranges follow the data slowly."""
        if self.training:
            m = self.ema_momentum
            lo = x.min() if k == 0 else torch.zeros(())
            self.act_lo[k] = (1 - m) * self.act_lo[k] + m * lo
            self.act_hi[k] = (1 - m) * self.act_hi[k] + m * x.max()
        scale, zp = self.act_params(k)
        return fake_quantize(x, scale, zp, -(2 ** (self.act_bits - 1)), 2 ** (self.act_bits - 1) - 1)

    def forward(self, x):
        if not self.calibrated:
            raise RuntimeError("call calibrate(X_train) first")
        x = self._act(0, x)                                         # quantize the input
        qmax_w = 2 ** (self.weight_bits - 1) - 1
        n = len(self.mlp.layers)
        for i, layer in enumerate(self.mlp.layers):
            s_w = self.weight_scale(layer)
            w = fake_quantize(layer.weight, s_w[:, None], 0, -qmax_w, qmax_w)
            s_x, _ = self.act_params(i)
            b = fake_quantize(layer.bias, s_x * s_w, 0, -(2 ** 31), 2 ** 31 - 1)   # int32 bias
            x = F.linear(x, w, b)
            if i < n - 1:
                x = self._act(i + 1, torch.relu(x))                 # ReLU, then quantize again
        return x

In [ ]:
ptq = QuantMLP(copy.deepcopy(baseline), weight_bits=8, act_bits=8)
ptq.calibrate(data.X_train)        # measure activation ranges on the training data

for k in range(len(ptq.act_lo)):
    scale, zp = ptq.act_params(k)
    name = "input" if k == 0 else f"after hidden layer {k}"
    print(f"{name:22s} range [{ptq.act_lo[k]:6.2f}, {ptq.act_hi[k]:6.2f}]  ->  scale {scale:.4f}, zero-point {zp}")

ptq_acc = accuracy(ptq, data.X_test, data.y_test)
print(f"\nPTQ int8 (simulated) test accuracy: {ptq_acc:.1%}   (float32: {base_acc:.1%})")

## 5. How does a layer compute with integers?

Write each real number as $r = s\,(q - z)$. For one neuron $j$ of a layer, with input scale $s_x$, zero-point $z_x$, weight scale $s_{w,j}$ (and weight zero-point 0):

$$y_j = \sum_i w_{ji} x_i + b_j = \sum_i s_{w,j} q_{w,ji} \cdot s_x (q_{x,i} - z_x) + b_j = s_{w,j} s_x \underbrace{\Big[\, q_{b,j} + \sum_i q_{w,ji}(q_{x,i} - z_x) \Big]}_{\text{acc}_j\text{: pure integer maths!}}$$

where the bias is stored as a 32-bit integer $q_{b,j} = \text{round}\big(b_j / (s_{w,j} s_x)\big)$. The expensive part, the sum with all the MACs, uses only integer multiplications and additions. Then we convert to the output's int8 representation:

$$q_{y,j} = \text{clamp}\big(\text{round}(M_j \cdot \text{acc}_j) + z_y,\ z_y,\ 127\big) \qquad M_j = \frac{s_{w,j}\, s_x}{s_y}$$

Two neat details:
* The lower clamp at $z_y$ **is the ReLU**: $z_y$ is where real 0 lies, so anything below it was negative.
* $M_j$ is one float multiplication per *neuron*, not per weight, so it costs almost nothing. (Chips without an FPU replace it by an integer multiplication plus a bit shift; see 7e.)

The accumulator must be 32 bits wide: 300 products of up to 127 × 255 add up to millions.

`int8_forward` implements exactly these formulas with NumPy integers. It is the twin of the C++ code that will run on the Arduino (notebook 05 checks that both give *identical* results).

`export_int8` (below) collects everything the board needs from the calibrated model: the **int8 weights**, the **int32 biases**, the (scale, zero-point) of every layer and the multiplier $M_j$ from the formula above. `int8_forward` then runs the network with NumPy *integers*, exactly like the C++ code.

Two small details in `int8_forward`:
* `round_half_away` rounds like C's `roundf` (2.5 → 3, −2.5 → −3). Python's own `round` would round 2.5 to 2, and the results would no longer match the board bit for bit.
* The `assert` checks that the accumulator stays small enough to be converted to float32 without any rounding.

In [ ]:
@torch.no_grad()
def export_int8(qmodel):
    """Collect the integer model: int8 weights, int32 biases, scales, zero-points and multipliers."""
    layers = []
    n = len(qmodel.mlp.layers)
    qmax_w = 2 ** (qmodel.weight_bits - 1) - 1
    for i, layer in enumerate(qmodel.mlp.layers):
        s_x, z_x = qmodel.act_params(i)                       # scale / zero-point of this layer's input
        s_w = qmodel.weight_scale(layer).double()             # one weight scale per neuron
        w_q = quantize(layer.weight.double(), s_w[:, None], 0, -qmax_w, qmax_w)
        b_q = torch.round(layer.bias.double() / (s_x * s_w))  # bias scale = s_x * s_w
        entry = {"w_q": w_q.numpy().astype(np.int8),
                 "b_q": b_q.numpy().astype(np.int32),
                 "in_zero_point": int(z_x)}
        if i < n - 1:
            s_y, z_y = qmodel.act_params(i + 1)               # scale / zero-point of the output
            entry["out_zero_point"] = int(z_y)
            entry["multiplier"] = (s_x * s_w.numpy() / s_y).astype(np.float32)   # M_j
        else:
            entry["multiplier"] = (s_x * s_w.numpy()).astype(np.float32)         # last layer: real logits
        layers.append(entry)
    s_in, z_in = qmodel.act_params(0)
    return {"layers": layers, "input_scale": np.float32(s_in), "input_zero_point": int(z_in)}


def round_half_away(v):
    """Round like C's roundf: halves go away from zero (2.5 -> 3, -2.5 -> -3)."""
    v = v.astype(np.float64)
    return np.sign(v) * np.floor(np.abs(v) + 0.5)


def int8_forward(int8_model, x):
    """int8 inference: integer MACs, one float rescale per neuron. x: normalized float features (N, 300). Returns float logits (N, 5)."""
    # quantize the input to int8
    a = round_half_away(np.asarray(x, dtype=np.float32) / int8_model["input_scale"]) + int8_model["input_zero_point"]
    a = np.clip(a, -128, 127).astype(np.int64)
    layers = int8_model["layers"]
    for i, L in enumerate(layers):
        # acc = bias + sum of w_q * (x_q - z_x): pure integer multiply-accumulate
        acc = L["b_q"].astype(np.int64) + (a - L["in_zero_point"]) @ L["w_q"].astype(np.int64).T
        assert np.abs(acc).max() < 2 ** 24, "accumulator too large for exact float32 conversion"
        p = acc.astype(np.float32) * L["multiplier"]          # one float multiply per neuron
        if i < len(layers) - 1:
            q = round_half_away(p) + L["out_zero_point"]
            a = np.clip(q, L["out_zero_point"], 127).astype(np.int64)   # the lower clamp is the ReLU
        else:
            return p.astype(np.float32)


def int8_accuracy(int8_model, X, y):
    return float((int8_forward(int8_model, X.numpy()).argmax(1) == y.numpy()).mean())


def int8_model_bytes(int8_model):
    """Flash memory: int8 weights + int32 biases + one float32 multiplier per neuron."""
    return sum(L["w_q"].size + 4 * L["b_q"].size + 4 * L["multiplier"].size for L in int8_model["layers"])

In [ ]:
int8_model = export_int8(ptq)
L0 = int8_model["layers"][0]
print("layer 0 weights :", L0["w_q"].dtype, L0["w_q"].shape, " e.g.", L0["w_q"][0, :8])
print("layer 0 biases  :", L0["b_q"].dtype, L0["b_q"].shape, " e.g.", L0["b_q"][:4])
print("layer 0 M       :", L0["multiplier"].dtype, " e.g.", L0["multiplier"][:3])

logits_int = int8_forward(int8_model, data.X_test.numpy())
with torch.no_grad():
    logits_sim = ptq(data.X_test).numpy()
print(f"\nint8 inference accuracy: {int8_accuracy(int8_model, data.X_test, data.y_test):.1%}")
print(f"same predictions as the simulation: {(logits_int.argmax(1) == logits_sim.argmax(1)).mean():.1%}"
      f"   (largest logit difference {np.abs(logits_int - logits_sim).max():.1e})")
print(f"memory: {int8_model_bytes(int8_model) / 1024:.1f} KB  (float32: {4 * count_params(baseline) / 1024:.1f} KB)")

So the int8 network is (practically) **as accurate as the float network, at ~1/4 of the memory**. The few extra bytes on top of a pure 1/4 come from the int32 biases and one float multiplier per neuron.

## 6. Fewer bits and quantization-aware training (QAT)

At 8 bits, PTQ is enough. But what if we want 4, 3 or even 2 bits? We saw that PTQ breaks down there. The fix: let the network **experience the rounding during training**, so it can learn weights that are robust to it.

**Quantization-aware training** (QAT):
* In the **forward pass** we use *fake quantization*: weights and activations are quantized and immediately dequantized, so the network sees exactly the rounding errors it will see on the device.
* In the **backward pass** there is a problem: `round()` is a staircase, so its gradient is 0 almost everywhere, and nothing would be learned. The trick is the **straight-through estimator (STE)**: during backpropagation we *pretend* the rounding was not there and pass the gradient straight through. In PyTorch this is one line: `x + (fake_quant(x) - x).detach()`.
* The underlying float weights are updated normally; they "drift" to values that still work well after rounding.

We start from the trained baseline and fine-tune for a few epochs with fake quantization. To be fair, we also fine-tune the *float* model for the same number of epochs (the "control"), so we can see what part of the improvement is just extra training. In all cases the activations stay 8-bit.

In [ ]:
bit_widths = [8, 4, 3, 2]
results = {"PTQ": [], "QAT": []}
set_seed(0)
control = copy.deepcopy(baseline)
train(control, data, lr=5e-4, epochs=30, patience=10, augment=True, verbose=False)
control_acc = accuracy(control, data.X_test, data.y_test)

for bits in bit_widths:
    q = QuantMLP(copy.deepcopy(baseline), weight_bits=bits)
    q.calibrate(data.X_train)
    results["PTQ"].append(accuracy(q, data.X_test, data.y_test))
    train(q, data, lr=5e-4, epochs=30, patience=10, augment=True, verbose=False)   # <- this is QAT
    results["QAT"].append(accuracy(q, data.X_test, data.y_test))
    print(f"{bits} bits: PTQ {results['PTQ'][-1]:.1%}   QAT {results['QAT'][-1]:.1%}")
print(f"float32 baseline {base_acc:.1%}, float32 fine-tuned (control) {control_acc:.1%}")

In [ ]:
fig, ax = plt.subplots(figsize=(6, 3.8))
for name, marker in zip(results, ["o", "s"]):
    ax.plot(bit_widths, results[name], marker=marker, markersize=6, label=name)
ax.axhline(base_acc, color="gray", ls=":", lw=1.5)
ax.text(bit_widths[-1], base_acc, "  float32 baseline", va="bottom", color="gray")
ax.set_xlabel("weight bits"); ax.set_ylabel("test accuracy"); ax.set_title("Accuracy vs weight bit width")
ax.legend(); ax.invert_xaxis(); plt.show()

**What do you see?**
* At **8 and 4 bits**, PTQ, QAT and the float **control** are all within about a point of each other: noise (remember notebook 01!). Any small advantage of QAT over PTQ here could just as well come from the extra training epochs. That's what the control measures. For int8, PTQ is all you need.
* At **2 bits**, PTQ collapses, while QAT brings the accuracy back to within about a point of the float models. The network has *learned* to live with only 3 different weight values per neuron (−1, 0, +1 times a scale).

This is why we included the control. Without it we might have concluded that QAT improves 8-bit models!

### Memory summary

In [ ]:
n_w = sum(l.weight.numel() for l in baseline.layers)
n_b = sum(l.bias.numel() for l in baseline.layers)
print(f"{'format':<34}{'weight bytes':>13}")
print(f"{'float32':<34}{4 * (n_w + n_b):>13,}")
for bits in [8, 4, 2]:
    # biases (int32) + one float multiplier per neuron stay 32-bit
    label = f"int{bits} weights" + (" (2 per byte)" if bits == 4 else " (4 per byte)" if bits == 2 else "")
    print(f"{label:<34}{n_w * bits // 8 + 8 * n_b:>13,}")

Note that 4- and 2-bit weights only save memory if you **pack** several of them into one byte, and then the C code must unpack them again (a bit of extra work per weight). The Arduino code in this course uses int8: the standard choice, supported directly by the hardware.


## 7. Other flavours of quantization

What we built in sections 4 and 5 is **full-integer post-training quantization**: int8 weights *and* int8 activations, with the activation ranges found by calibration. All the MACs run on integers; only the rescaling by $M_j$ (section 5) is one float multiply per neuron. That is cheap on our Cortex-M4F, which has a floating-point unit. Toolkit kernels do the rescaling with integers too (see 7e), so their models run **integer-only**, also on chips without floating-point hardware.

Toolkits such as TensorFlow Lite / LiteRT (its `TFLiteConverter`) and PyTorch offer a small menu of other variants. You will meet their names in tutorials and papers, so here is what each one does:

| Variant | Weights stored as | Activations computed as | Calibration data? | Integer-only? | Model size | Typical speed-up on a CPU* | Typical use |
|---|---|---|---|---|---|---|---|
| **Dynamic-range** | int8 | float; quantized to int8 *on the fly* for every input | no | no | ~4x smaller | 2-3x | a quick first try on phones and PCs |
| **Full integer** (this notebook) | int8 | int8 | yes (~100-500 examples) | **yes** in toolkits; ours: integer MACs, float rescaling | ~4x smaller | 3x or more | microcontrollers, Edge TPU, NPUs |
| **float16** | float16 | float16 or float32 | no | no | 2x smaller | none on a CPU; fast on GPUs | GPUs |
| **16x8** | int8 | int16 | yes | yes | ~3-4x smaller | slower than int8 | audio, image clean-up: when activations need precision |
| **QAT** (section 6) | any of the above | | the training data | | | | when post-training quantization loses too much accuracy |

\* Rough numbers from the TensorFlow Lite documentation. They depend a lot on the hardware.

All variants except QAT are *post-training*: they work on an already trained model, without any training. The next sections try them on our baseline.

### 7a. Dynamic-range quantization

Only the **weights** are quantized ahead of time. The activations stay float, so **no calibration** is needed. Instead, every time a layer runs, the program looks at the actual input values of *this* prediction, picks a scale for them on the spot, quantizes them, does the MACs with integers and converts the result back to float. The bias, the ReLU and everything else stay float.

* **+** No representative data needed, very easy to try, and the expensive MACs still run on integers.
* **−** Measuring the range at run time costs extra work in every layer, and the chip still needs float arithmetic. Not usable on integer-only hardware.

Below is a simple version (symmetric, one activation scale per example). For readability it re-computes the int8 weights on every call; a real model stores them once.

In [ ]:
@torch.no_grad()
def dynamic_range_forward(model, x, bits=8):
    """int8 weights; each layer's input is quantized on the fly, everything else stays float."""
    qmax = 2 ** (bits - 1) - 1
    n = len(model.layers)
    for i, layer in enumerate(model.layers):
        s_w = layer.weight.abs().amax(dim=1) / qmax                       # weight scales: fixed, stored with the model
        w_q = torch.round(layer.weight / s_w[:, None])                    # int8 weights: fixed, stored with the model
        s_x = x.abs().amax(dim=1, keepdim=True).clamp(min=1e-8) / qmax    # input scale: measured NOW, per example
        x_q = torch.round(x / s_x)                                        # int8 input of this layer
        acc = x_q @ w_q.T                                                 # the MACs: integers only
        x = acc * s_x * s_w + layer.bias                                  # back to float, float bias
        if i < n - 1:
            x = torch.relu(x)                                             # float ReLU
    return x

dyn_acc = float((dynamic_range_forward(baseline, data.X_test).argmax(1) == data.y_test).float().mean())
print(f"dynamic-range int8: test accuracy {dyn_acc:.1%}   (full-integer PTQ {ptq_acc:.1%}, float32 {base_acc:.1%})")

About as accurate as full-integer PTQ, and we did not need a single calibration example. The weights take 1 byte each, so the model is just as small. The difference is at run time. Our full-integer version needs one float multiply per neuron, by a multiplier $M_j$ that is fixed after calibration, so a toolkit can turn it into integer steps (see 7e). Dynamic-range quantization measures each input's range during every prediction, so its scales change all the time, and the bias, the ReLU and the rescaling all stay float. That's why microcontrollers use full-integer quantization.

### 7b. float16

**float16** ("half precision") is a 16-bit floating-point number: 1 sign bit, 5 exponent bits and 10 mantissa bits (float32 has 1, 8 and 23). Converting a model just means storing every weight with fewer digits: no scale, no zero-point, no calibration.

* A float16 keeps only about **3 significant decimal digits**.
* The largest float16 is **65 504**; anything bigger becomes infinity (`inf`). Very small numbers become 0.

For weights (small numbers, a few digits are enough) that is no problem: the model gets **2x smaller** and the accuracy practically does not change. GPUs, and some phone chips, compute in float16 directly and fast. Our Cortex-M4F has no float16 hardware: the weights would be converted back to float32 before use. A smaller file, but no speed gain.

In [ ]:
print("float32 -> float16:", torch.tensor([0.1, 3.14159, 1000.3, 70000.0, 1e-8]).half())

m16 = copy.deepcopy(baseline)
with torch.no_grad():
    for layer in m16.layers:
        layer.weight.copy_(layer.weight.half().float())    # round to float16 and back
        layer.bias.copy_(layer.bias.half().float())
err = max((a.weight - b.weight).abs().max().item() for a, b in zip(m16.layers, baseline.layers))
print(f"\nfloat16 weights: test accuracy {accuracy(m16, data.X_test, data.y_test):.1%} (float32 {base_acc:.1%}), "
      f"largest weight change {err:.1e}, size {2 * count_params(baseline) / 1024:.1f} KB "
      f"(float32 {4 * count_params(baseline) / 1024:.1f} KB)")

### 7c. 16x8: 8-bit weights, 16-bit activations

Sometimes 256 levels are not enough for the *activations*. That is typical for networks whose output is itself a signal: noise cancelling, beamforming with a microphone array, super-resolution (making images sharper), de-noising images, or HDR reconstruction. A small rounding error in every output sample is then audible or visible.

The **16x8** scheme keeps int8 weights, so the model is still about 4x smaller than float32, but uses **int16 activations**: 65 536 levels instead of 256. Like full-integer int8 in the toolkits, it can run integer-only. Two rules differ from our int8 version: the activations are **symmetric** (zero-point 0), and the biases are stored as **int64**. The price:
* activations take twice as much RAM;
* wider sums: one int16 × int8 product can be as large as 32 768 × 128 ≈ 4.2 million, so about 512 of those worst-case products already fill a 32-bit integer. Our layers (at most 300 inputs) stay below that, but a convolution layer easily has thousands of inputs. The bias grows too: its scale $s_x \cdot s_w$ is more than 100x finer than with int8 activations, so the same real bias becomes an integer more than 100x larger. That is why the toolkits use 64-bit biases and accumulators for 16x8;
* fewer chips have fast int16 × int8 instructions, so it is usually slower than plain int8.

Below, `QuantMLP16x8` simulates it. It is our `QuantMLP` with one change: a symmetric activation scale with zero-point 0. After the ReLU the activations are never negative, so only the positive half of the levels gets used, still 32 768 of them. The biases stay int32 in `QuantMLP`; the last line checks that none of them comes close to that limit, so 64-bit biases would give the same result here. It is a simulation with fake quantization, not the toolkit's converter.

In [ ]:
class QuantMLP16x8(QuantMLP):
    """QuantMLP with the 16x8 activation rule: symmetric scale, zero-point 0."""

    def act_params(self, k):
        qmax = 2 ** (self.act_bits - 1) - 1
        r = max(abs(self.act_lo[k].item()), abs(self.act_hi[k].item()), 1e-8)
        return r / qmax, 0

for label, cls, act_bits in [("int8 ", QuantMLP, 8), ("int16", QuantMLP16x8, 16)]:
    q = cls(copy.deepcopy(baseline), weight_bits=8, act_bits=act_bits)
    q.calibrate(data.X_train)
    act_ram = (300 + 64) * act_bits // 8          # the input and the largest hidden layer, at the same time
    print(f"weights int8, activations {label}: test accuracy {accuracy(q, data.X_test, data.y_test):.1%}, "
          f"activation RAM ~{act_ram} bytes")

# q is now the 16x8 model: how large do its integer biases get?
largest_bias = max((layer.bias / (q.act_params(i)[0] * q.weight_scale(layer))).abs().max().item()
                   for i, layer in enumerate(q.mlp.layers))
print(f"largest integer bias with int16 activations: {largest_bias:,.0f}   (int32 limit {2 ** 31 - 1:,})")

For a classifier like ours, 16-bit activations bring nothing: the largest logit hardly changes because of tiny rounding errors. Choose 16x8 when the *exact output values* matter, not just which one is largest.

### 7d. Quantizing only part of the network

Not every layer is equally sensitive. If one layer suffers badly from quantization, you can keep it at higher precision and quantize the rest: **partial** or **mixed-precision** quantization. First, measure the sensitivity: quantize **one layer at a time** to 2 bits and keep the others in float.

In [ ]:
def quantize_one_layer(model, layer_index, bits):
    """Copy of `model` with only one layer's weights rounded to `bits` bits; the others stay float."""
    m = quantize_weights(model, bits)                          # all layers rounded (section 3) ...
    with torch.no_grad():
        for i, layer in enumerate(m.layers):
            if i != layer_index:
                layer.weight.copy_(model.layers[i].weight)     # ... then put back the float weights of the others
    return m

for i, layer in enumerate(baseline.layers):
    acc = accuracy(quantize_one_layer(baseline, i, bits=2), data.X_test, data.y_test)
    print(f"only layer {i} ({layer.in_features:3d} -> {layer.out_features:2d}, {layer.weight.numel():6,} weights) at 2 bits: {acc:.1%}")
print(f"all layers at 2 bits: {accuracy(quantize_weights(baseline, 2), data.X_test, data.y_test):.1%}   (float32 {base_acc:.1%})")

The layers clearly differ. Here the big first layer usually hurts most when it is rounded to 2 bits, and it also holds about 90% of all weights, so keeping it at high precision would undo most of the savings. A mixed plan could be, for example, 4 bits for the first layer and 2 bits for the two small ones. In general: measure first, then spend the bits where they matter.

**Partial quantization-aware training.** The same idea works during training. You mark the layers that should be quantized, and only those get fake quantization during fine-tuning. In TensorFlow you wrap them with `quantize_annotate_layer` and then call `quantize_apply`; quantizing the whole model at once is `quantize_model`. Our `QuantMLP` always quantizes all layers; making `weight_bits` a per-layer list would be the change.

What it costs on the device:
* at every border between an integer layer and a float layer, the program must **dequantize** (integer → float) or **quantize** (float → integer) the activations;
* the float layers need floating-point hardware. That's fine on our Cortex-M4F, but on integer-only accelerators those layers fall back to the main CPU and run slowly.

Typical reasons to keep a layer in float: it is very sensitive (often the first or the last layer), it has extreme outliers, or it uses an operation that has no good integer version.

### 7e. Integer inputs and outputs

Look again at `int8_forward`: it takes **float** features and returns **float** logits. Quantizing the input and dequantizing the logits happen inside, at the borders. TensorFlow Lite's converter does the same by default: integers inside, a float interface outside, so the quantized model is a drop-in replacement for the float one.

Some hardware can only handle integers, for example Google's Edge TPU and many NPUs (neural processing units). There you need three things that our version does not have:
1. **Integer inputs and outputs** (in TensorFlow Lite: `inference_input_type` and `inference_output_type` set to `int8` or `uint8`). The program then hands the model integers directly, for example raw camera bytes, and gets integer scores back. On our board you could merge the normalisation and the input quantization into one step: straight from the raw sensor value to int8.
2. **Integer rescaling.** Integer inputs and outputs alone do not make `int8_forward` integer-only: between the layers it still multiplies by the float $M_j$. Toolkits write each $M_j$ as an integer times a power of two, $M_j \approx m_j \cdot 2^{-n_j}$, with $m_j$ a 32-bit integer. The rescaling then becomes an integer multiply followed by a right shift by $n_j$ bits. Because $M_j$ is fixed after calibration, $m_j$ and $n_j$ are computed once, at conversion time. (Our Arduino code keeps the float multiply: the Cortex-M4F has a floating-point unit, and the float version is easier to read.)
3. A **strict conversion**: if any operation cannot be done with integers, the conversion must *fail* (in TensorFlow Lite: `supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]`). Otherwise the model looks quantized, but one float operation quietly runs on the slow main CPU.

**int8 or uint8?** It is the same ruler, shifted by 128: uint8 runs from 0 to 255, and $q_{uint8} = q_{int8} + 128$ (the zero-point shifts by 128 too). Images are usually stored as uint8 pixels (0-255), which is why camera models often have uint8 inputs.

## Summary

* Quantization maps real numbers to integers: $q = \text{round}(r/s) + z$, $r \approx s\,(q - z)$.
* Weights: symmetric, one scale per neuron (per-channel). Activations: asymmetric, ranges found by **calibration**.
* With the right formulas a whole layer runs on **integer MACs** plus one float multiply per neuron.
* **int8 PTQ** shrinks the model 4x at essentially no accuracy cost.
* At very low bit widths, **QAT** (fake quantization + straight-through estimator) recovers the accuracy.
* **Dynamic-range** quantization: int8 weights, activations quantized on the fly; no calibration needed, but float maths remains.
* **float16**: 2x smaller, no scales at all; useful on GPUs, not on our board.
* **16x8**: int8 weights with int16 activations, for networks whose exact output values matter (audio, images).
* **Partial / mixed-precision** quantization: measure which layers are sensitive and keep those at higher precision.
* Integer-only hardware also needs **integer rescaling** (an integer multiply and a shift instead of our float multiply), **integer inputs and outputs**, and a **strict** conversion that fails instead of silently leaving an operation in float.

## Exercises

1. ⭐ **By hand.** Quantize the weights `[0.9, -0.3, 0.02, -1.8]` to 4 bits (symmetric, $q \in [-7, 7]$) on paper. What are $s$, the $q$'s, the dequantized values and the largest error? Check with code.
2. ⭐ **Outliers.** Take a copy of the baseline and set *one* weight of layer 0 to 20.0. Compare per-tensor vs per-channel 4-bit accuracy (`quantize_weights`). Explain the result with the "ruler" picture.
3. **Activation bits.** Try `QuantMLP(..., act_bits=4)` with weights at 8 bits. (Note: `export_int8` and the board only supports 8-bit activations, but the simulation works.) Which is more sensitive, weights or activations?
4. **Calibration data.** Calibrate the PTQ model on only the first 10 training examples instead of all. What happens to the ranges and the accuracy? Why?
5. **Smaller models.** Repeat the bit-width experiment for a small `MLP(hidden=(16, 8))`. Do small networks tolerate quantization better or worse? Why could that be?